# 🩸 SampleCheck
## Blood Transfusion Sample Rejection Audit

**A guided Jupyter Notebook for exploring rejected-sample records**

---

### What this notebook helps you do

Use an Excel spreadsheet of rejected samples to:

- check whether the data is complete and consistently recorded;
- count and compare rejection reasons;
- explore patterns by sample type and date, where those fields are available;
- create charts and export a summary workbook.

> **Important:** This notebook is an analytical aid, not a replacement for laboratory procedures, clinical judgement, incident reporting or local policy. It does not decide whether a sample should be accepted or rejected.

**Working principle:** inspect first, flag issues, investigate them, and only then decide whether any correction is appropriate. The original input file is never overwritten.

## How to use this notebook

1. Save this notebook in a working folder on an approved computer.
2. Put your Excel file in the same folder, or update the file path in **Configuration**.
3. Check the column names in your spreadsheet and update the mapping below.
4. Run the notebook from top to bottom using **Run All** (or run each cell in order).
5. Review the data-quality flags before interpreting the charts.
6. Find the Excel report in the same folder when the final export cell completes.

**Data protection:** use only an environment and data that your organisation has approved. Keep patient-identifiable information out of the analysis unless its use is explicitly authorised and necessary. This notebook does not upload data, but local processing alone does not establish approval. Do not email or share the output outside approved channels.

## 1. Configuration

This is the main settings cell. Most users should only need to check the file name and column mapping.

**Expected structure:** one row represents one rejected-sample record, and one rejection reason is recorded in each row. If your local spreadsheet uses a different structure (for example, several rejection reasons per sample), adapt the analysis before using it.

The category list below is editable. Use the wording agreed by your laboratory so that categories are recorded consistently.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["figure.dpi"] = 110

# ---- Input and output files ----
INPUT_FILE = Path("sample_rejections.xlsx")
SHEET_NAME = 0                 # 0 = first worksheet; or use a worksheet name
OUTPUT_FILE = Path("SampleCheck_Report.xlsx")

# ---- Map these to the exact column names in your Excel file ----
REASON_COLUMN = "Rejection Reason"
SAMPLE_TYPE_COLUMN = "Sample Type"   # optional; set to None if unavailable
DATE_COLUMN = "Date"                 # optional; set to None if unavailable
RECORD_ID_COLUMN = None              # optional unique record/sample ID column

# ---- Standard rejection categories (edit to match local terminology) ----
REJECTION_CATEGORIES = [
    "Sample received unlabelled",
    "Hospital number error",
    "Date of Birth error",
    "Sample tube expired",
    "Duplicate sample received",
    "Clotted Sample",
    "Sample received without EPIC label",
]

# Matching ignores leading/trailing spaces and letter case.
# It does not silently rewrite the source data.

### 1.1 Check the configuration

Before continuing, confirm:

- the input file name is correct;
- the rejection-reason column exists in the spreadsheet;
- optional columns are mapped correctly, or set to `None`;
- the category list reflects the terminology used in your department.

If the file is not in the notebook's folder, provide its full or relative path in `INPUT_FILE`.

In [ ]:
print("Notebook working folder:", Path.cwd())
print("Input file:", INPUT_FILE.resolve())
print("Input file exists:", INPUT_FILE.exists())
print("Configured rejection categories:")
for number, category in enumerate(REJECTION_CATEGORIES, start=1):
    print(f"{number}. {category}")

if not INPUT_FILE.exists():
    print("\nACTION REQUIRED: Put the Excel file in this folder or update INPUT_FILE.")

## 2. Import the Excel data

The next cell reads the selected worksheet. The source workbook is read only; this notebook does not modify it.

If the file contains multiple worksheets, check `SHEET_NAME` in Configuration.

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {INPUT_FILE}. Check the file name and folder in Configuration."
    )

raw_df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(f"Rows loaded: {len(raw_df):,}")
print(f"Columns found: {len(raw_df.columns)}")
display(raw_df.head())

### 2.1 Understand the structure

Check the column names, data types and dimensions. This helps identify mismatches between the spreadsheet and the configuration.

Do not assume that a blank cell means zero, or that two identical-looking rows are necessarily erroneous.

In [ ]:
print("Shape (rows, columns):", raw_df.shape)
print("\nColumn names:")
print(raw_df.columns.tolist())
print("\nData types:")
display(raw_df.dtypes.to_frame("Data type"))
print("\nMissing values by column:")
display(raw_df.isna().sum().to_frame("Missing values"))

## 3. Validate the configured columns

The rejection-reason column is essential. Sample type and date are optional and will only be analysed when configured and present.

In [ ]:
if REASON_COLUMN not in raw_df.columns:
    raise KeyError(
        f"'{REASON_COLUMN}' was not found. Available columns: {raw_df.columns.tolist()}"
    )

for optional_column, label in [
    (SAMPLE_TYPE_COLUMN, "Sample type"),
    (DATE_COLUMN, "Date"),
    (RECORD_ID_COLUMN, "Record ID"),
]:
    if optional_column is not None and optional_column not in raw_df.columns:
        print(f"NOTE: {label} column '{optional_column}' was not found; related analysis will be skipped.")

## 4. Data-quality audit

This section flags issues for review. It does **not** delete rows, remove duplicates or replace missing values.

### 4.1 Missing and unrecognised rejection reasons

A missing reason is different from a reason that is present but not in the configured category list. Both should be reviewed.

In [ ]:
audit_df = raw_df.copy()

def normalise_label(series):
    return series.astype("string").str.strip().str.casefold()

category_lookup = {category.strip().casefold(): category for category in REJECTION_CATEGORIES}
reason_text = audit_df[REASON_COLUMN].astype("string").str.strip()
reason_key = normalise_label(audit_df[REASON_COLUMN])

audit_df["Reason missing"] = audit_df[REASON_COLUMN].isna() | reason_text.eq("")
audit_df["Reason recognised"] = reason_key.isin(category_lookup)
audit_df["Reason not recognised"] = (~audit_df["Reason missing"]) & (~audit_df["Reason recognised"])

quality_summary = pd.DataFrame({
    "Check": [
        "Total rows",
        "Missing rejection reason",
        "Unrecognised rejection reason",
        "Recognised rejection reason",
    ],
    "Count": [
        len(audit_df),
        int(audit_df["Reason missing"].sum()),
        int(audit_df["Reason not recognised"].sum()),
        int(audit_df["Reason recognised"].sum()),
    ],
})
display(quality_summary)

### 4.2 Review unrecognised labels

Examples may include spelling differences, abbreviations or categories that have not yet been added to the configured list. Do not automatically map them: first confirm what the original entry means with the appropriate local team.

In [ ]:
unrecognised_values = (
    audit_df.loc[audit_df["Reason not recognised"], REASON_COLUMN]
    .value_counts(dropna=False)
    .rename_axis("Unrecognised value")
    .reset_index(name="Rows")
)
display(unrecognised_values if not unrecognised_values.empty else
        pd.DataFrame({"Message": ["No unrecognised rejection reasons detected."]}))

### 4.3 Check for possible duplicate records

Exact duplicate rows are flagged for review. If a record/sample ID is configured, repeated IDs are also shown.

**Important:** repeated IDs are not automatically errors. A sample may legitimately appear more than once depending on how the data is recorded. Confirm the record structure before deciding what to do.

In [ ]:
exact_duplicate_mask = audit_df.duplicated(keep=False)
print("Rows involved in exact duplicate groups:", int(exact_duplicate_mask.sum()))
display(audit_df.loc[exact_duplicate_mask].head(20))

if RECORD_ID_COLUMN is not None and RECORD_ID_COLUMN in audit_df.columns:
    repeated_id_mask = audit_df[RECORD_ID_COLUMN].duplicated(keep=False)
    print(f"Rows with a repeated {RECORD_ID_COLUMN}:", int(repeated_id_mask.sum()))
    display(audit_df.loc[repeated_id_mask, [RECORD_ID_COLUMN, REASON_COLUMN]].head(20))
else:
    print("No record ID column configured; ID-based duplicate check skipped.")

### 4.4 Check dates, where available

Dates are converted in a separate analysis column. Unparseable non-empty values are flagged; the original date column is preserved.

In [ ]:
date_ready = (
    DATE_COLUMN is not None
    and DATE_COLUMN in audit_df.columns
)

if date_ready:
    parsed_dates = pd.to_datetime(audit_df[DATE_COLUMN], errors="coerce", dayfirst=True)
    nonempty_original_dates = audit_df[DATE_COLUMN].notna() & audit_df[DATE_COLUMN].astype("string").str.strip().ne("")
    invalid_date_mask = nonempty_original_dates & parsed_dates.isna()
    audit_df["Analysis date"] = parsed_dates
    print("Missing date values:", int(audit_df[DATE_COLUMN].isna().sum()))
    print("Non-empty dates that could not be parsed:", int(invalid_date_mask.sum()))
    display(audit_df.loc[invalid_date_mask, [DATE_COLUMN]].head(20))
else:
    print("Date analysis skipped: no valid date column configured.")

## 5. Rejection-reason analysis

The analysis below uses records with a recognised rejection reason. Missing and unrecognised reasons are excluded from the category summary but remain in the audit output.

**Denominator warning:** percentages here represent each category's share of recognised rejection records. They are **not** the percentage of all samples received that were rejected. Calculating a true rejection rate requires a reliable denominator containing all samples received, including accepted samples.

In [ ]:
recognised_df = audit_df.loc[audit_df["Reason recognised"]].copy()
recognised_df["Standard rejection reason"] = reason_key.loc[recognised_df.index].map(category_lookup)

reason_counts = (
    recognised_df["Standard rejection reason"]
    .value_counts()
    .reindex(REJECTION_CATEGORIES, fill_value=0)
    .rename_axis("Rejection reason")
    .reset_index(name="Count")
)

total_recognised = int(reason_counts["Count"].sum())
reason_counts["Percentage of recognised rejections (%)"] = (
    reason_counts["Count"] / total_recognised * 100
    if total_recognised > 0 else 0
).round(1)

display(reason_counts)
print(f"Recognised rejection records included: {total_recognised:,}")

### 5.1 Visualise rejection reasons

The chart makes it easier to compare the volume of each reason. Categories with zero records remain visible so that the configured list can be checked.

In [ ]:
plot_data = reason_counts.sort_values("Count", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(plot_data["Rejection reason"], plot_data["Count"])
ax.set_title("Rejected samples by recorded rejection reason")
ax.set_xlabel("Number of recognised rejection records")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)

for y, value in enumerate(plot_data["Count"]):
    ax.text(value, y, f" {value}", va="center")

plt.tight_layout()
plt.show()

### 5.2 Read the results carefully

Use the table and chart to identify the most frequently recorded reasons. A high count may reflect a genuine process issue, differences in recording practice, workload or data completeness.

The analysis describes the records supplied; it does not by itself establish the cause of a rejection or prove that one department or staff group is performing poorly.

## 6. Compare rejection reasons by sample type

This comparison is available when a sample-type column has been configured and found.

The percentages are calculated **within each sample type**, among recognised rejection records for that type. They do not represent the rejection rate among all samples of that type unless accepted samples are also included in the source data.

In [ ]:
sample_type_ready = (
    SAMPLE_TYPE_COLUMN is not None
    and SAMPLE_TYPE_COLUMN in recognised_df.columns
)

if sample_type_ready:
    sample_type_data = recognised_df.copy()
    sample_type_data[SAMPLE_TYPE_COLUMN] = (
        sample_type_data[SAMPLE_TYPE_COLUMN].astype("string").str.strip()
    )
    sample_type_data[SAMPLE_TYPE_COLUMN] = sample_type_data[SAMPLE_TYPE_COLUMN].fillna("Missing sample type")

    type_counts = pd.crosstab(
        sample_type_data[SAMPLE_TYPE_COLUMN],
        sample_type_data["Standard rejection reason"]
    ).reindex(columns=REJECTION_CATEGORIES, fill_value=0)

    type_percentages = type_counts.div(type_counts.sum(axis=1).replace(0, np.nan), axis=0) * 100
    display(type_counts)
    display(type_percentages.round(1))

    ax = type_counts.plot(kind="bar", figsize=(12, 6))
    ax.set_title("Rejection reasons by sample type")
    ax.set_xlabel("Sample type")
    ax.set_ylabel("Recognised rejection records")
    ax.legend(title="Rejection reason", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Sample-type comparison skipped. Configure SAMPLE_TYPE_COLUMN to enable it.")

## 7. Explore patterns over time

If a date column is available, this section groups recognised rejection records by calendar month. It shows counts of recorded rejections, not rejection rates.

In [ ]:
if date_ready:
    trend_data = audit_df.loc[
        audit_df["Reason recognised"] & audit_df["Analysis date"].notna()
    ].copy()

    if not trend_data.empty:
        trend_data["Month"] = trend_data["Analysis date"].dt.to_period("M").astype(str)
        monthly_counts = trend_data.groupby("Month").size().rename("Recognised rejection records")
        display(monthly_counts.to_frame())

        ax = monthly_counts.plot(marker="o", figsize=(11, 4))
        ax.set_title("Recognised rejection records by month")
        ax.set_xlabel("Month")
        ax.set_ylabel("Number of rejection records")
        ax.grid(alpha=0.25)
        plt.xticks(rotation=35)
        plt.tight_layout()
        plt.show()
    else:
        print("No recognised rejection records with valid dates are available.")
else:
    print("Time trend skipped because no date column is configured.")

## 8. Write a cautious summary

The cell below generates descriptive statements from the data. Treat them as prompts for review, not clinical conclusions. Check the tables and data-quality flags before sharing any interpretation.

In [ ]:
if total_recognised > 0:
    top_reason = reason_counts.sort_values("Count", ascending=False).iloc[0]
    top_share = float(top_reason["Percentage of recognised rejections (%)"])
    print(f"The most frequently recorded recognised rejection reason was "
          f"'{top_reason['Rejection reason']}' ({int(top_reason['Count'])} records; "
          f"{top_share:.1f}% of recognised rejection records).")
else:
    print("No recognised rejection reasons were available for summary.")

print(f"\nTotal input rows: {len(audit_df):,}")
print(f"Rows with missing reasons: {int(audit_df['Reason missing'].sum()):,}")
print(f"Rows with unrecognised reasons: {int(audit_df['Reason not recognised'].sum()):,}")
print(f"Rows involved in exact duplicate groups: {int(exact_duplicate_mask.sum()):,}")

print("\nInterpretation reminder:")
print("- These are counts and shares of recorded rejection records.")
print("- They are not rejection rates among all samples received.")
print("- The analysis does not establish why a rejection occurred.")

## 9. Export the Excel report

The report contains:
- a summary of rejection reasons and their share of recognised rejection records;
- data-quality counts;
- unrecognised reason labels;
- the analysed records with review flags;
- sample-type counts and percentages, if available;
- monthly counts, if valid dates are available.

The source workbook is not changed. Review the report for unnecessary identifiers before sharing it, and follow local information-governance requirements.

In [ ]:
quality_export = quality_summary.copy()
unrecognised_export = unrecognised_values.copy()

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    reason_counts.to_excel(writer, sheet_name="Reason summary", index=False)
    quality_export.to_excel(writer, sheet_name="Data quality", index=False)
    unrecognised_export.to_excel(writer, sheet_name="Unrecognised labels", index=False)
    audit_df.to_excel(writer, sheet_name="Audited records", index=False)

    if sample_type_ready:
        type_counts.to_excel(writer, sheet_name="Counts by sample type")
        type_percentages.round(1).to_excel(writer, sheet_name="Percent by sample type")

    if date_ready and "Month" in trend_data.columns if date_ready and "trend_data" in globals() else False:
        monthly_counts.to_frame().to_excel(writer, sheet_name="Monthly trend")

print("Report created:", OUTPUT_FILE.resolve())
print("File size (bytes):", OUTPUT_FILE.stat().st_size)

## 10. Final review checklist

Before using or sharing the report, confirm:

- [ ] The spreadsheet represents the intended reporting period and population.
- [ ] The configured column names are correct.
- [ ] The rejection categories match local terminology.
- [ ] Missing and unrecognised reasons have been reviewed.
- [ ] Potential duplicates have been investigated rather than automatically deleted.
- [ ] The denominator behind each percentage is understood.
- [ ] Sample-type comparisons are not being mistaken for sample-type rejection rates.
- [ ] Patient-identifiable information is not included unnecessarily in the exported report.
- [ ] Findings have been checked with the appropriate laboratory lead before operational decisions are made.

---

### Key lesson

A notebook can make an audit repeatable, but it cannot make poor source data reliable by itself. The quality of the interpretation depends on understanding how each row was created, what each field means and which records are included in each denominator.

**End of SampleCheck notebook.**